In [ ]:
# STEP 1: IMPORTS & PATH SETUP
# Description: Load required libraries and define all file paths

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
models_root = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
vent_ref_root = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/ventilation"

# Year to simulate
YEAR = 2026

In [9]:
# STEP 2: LOAD AND CLEAN EPC DATA
# Description: Load EPC dataset and standardise ventilation column

epc_df = pd.read_csv(epc_path)

# Fill missing ventilation values with "natural"
epc_df["MECHANICAL_VENTILATION"] = epc_df["MECHANICAL_VENTILATION"].fillna("natural")

# Ensure consistent formatting
epc_df["MECHANICAL_VENTILATION"] = epc_df["MECHANICAL_VENTILATION"].str.strip().str.lower()

In [10]:
# STEP 3: GENERATE TIME COLUMN
# Description: Create 17520 half-hour timesteps starting at 00:30:00

start_time = datetime(YEAR, 1, 1, 0, 30)
timesteps = 17520  # 365 days * 48

time_list = [(start_time + timedelta(minutes=30*i)).strftime("%H:%M:%S") for i in range(timesteps)]

In [11]:
# STEP 4: CREATE DAY TYPE ARRAY
# Description: Build a list matching each timestep to wd/we

day_types = []

current_date = datetime(YEAR, 1, 1)

for day in range(365):
    is_weekend = current_date.weekday() >= 5  # 5=Sat, 6=Sun
    day_label = "we" if is_weekend else "wd"
    
    # Repeat 48 times (half-hour intervals)
    day_types.extend([day_label]*48)
    
    current_date += timedelta(days=1)

# Sanity check
assert len(day_types) == 17520

In [12]:
# STEP 5: DEFINE VENTILATION TYPE LOGIC
# Description: Map EPC ventilation types to schedule types

def get_vent_type(epc_value):
    if epc_value == "extract only":
        return "extract"
    elif epc_value == "mechanical, supply and extract":
        return "continuous"
    else:
        return None  # natural → skip

In [13]:
# STEP 6: MAIN LOOP THROUGH BUILDINGS (UPDATED)
# Description: Generate ventilation schedule OR delete file if natural ventilation

for _, row in epc_df.iterrows():
    
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    archetype = row["ARCHETYPE"]
    vent_type_raw = row["MECHANICAL_VENTILATION"]
    
    vent_type = get_vent_type(vent_type_raw)
    
    # Paths
    building_folder = os.path.join(models_root, building_id)
    vent_folder = os.path.join(building_folder, "VENTILATION")
    output_path = os.path.join(vent_folder, "ventilation_fan_power.csv")
    
    # --- NEW LOGIC: HANDLE NATURAL VENTILATION ---
    if vent_type is None:
        if os.path.exists(output_path):
            os.remove(output_path)
            print(f"Deleted (natural ventilation): {output_path}")
        else:
            print(f"Skipped (natural ventilation, no file): {building_id}")
        continue
    
    # Reference file path
    ref_file = os.path.join(vent_ref_root, f"{archetype}.csv")
    
    # Check reference file exists
    if not os.path.exists(ref_file):
        print(f"Skipping {building_id} — missing archetype file: {archetype}")
        continue
    
    # Ensure ventilation folder exists
    os.makedirs(vent_folder, exist_ok=True)
    
    # Load archetype schedule
    ref_df = pd.read_csv(ref_file)
    
    # Extract correct columns
    wd_col = f"{vent_type}_wd"
    we_col = f"{vent_type}_we"
    
    wd_values = ref_df[wd_col].values
    we_values = ref_df[we_col].values
    
    # Build full-year schedule
    yearly_values = []
    
    for i in range(17520):
        if day_types[i] == "wd":
            yearly_values.append(wd_values[i % 48])
        else:
            yearly_values.append(we_values[i % 48])
    
    # Create DataFrame
    output_df = pd.DataFrame({
        "Time": time_list,
        "ventilation_fan_power": yearly_values
    })
    
    # Save (overwrite if exists)
    output_df.to_csv(output_path, index=False)
    
    print(f"Saved: {output_path}")

Skipped (natural ventilation, no file): 1001829067
Skipped (natural ventilation, no file): 1001951858
Skipped (natural ventilation, no file): 1001829071
Skipped (natural ventilation, no file): 1234761001
Skipped (natural ventilation, no file): 1001991633
Skipped (natural ventilation, no file): 1001664929
Skipped (natural ventilation, no file): 1001829059
Skipped (natural ventilation, no file): 1001063639
Skipped (natural ventilation, no file): 1234761000
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1236594950/VENTILATION/ventilation_fan_power.csv
Skipped (natural ventilation, no file): 1001664925
Skipped (natural ventilation, no file): 1001906271
Skipped (natural ventilation, no file): 1000238911
Skipped (natural ventilation, no file): 1001829057
Skipped (natural ventilation, no file): 1234760999
Skipped (natural ventilation, no file): 1002143357
Skipped (natural ventilation, no file): 1001951854
Skipped (natural ventilation, no file): 1001829069
Skipped (natural ventilat

In [14]:
# STEP 7: QUICK VALIDATION
# Description: Confirm outputs look correct for a sample building

sample_building = epc_df.iloc[0]["BUILDING_REFERENCE_NUMBER"]
sample_path = os.path.join(models_root, str(sample_building), "VENTILATION", "ventilation_fan_power.csv")

if os.path.exists(sample_path):
    df_check = pd.read_csv(sample_path)
    print(df_check.head())
    print("Rows:", len(df_check))
else:
    print("Sample file not found.")

Sample file not found.
